In [33]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/cdeotte/s6e4-the-original-dataset/irrigation_prediction.csv
/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


# Load Data and Inspect

In [34]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")
orig = pd.read_csv("/kaggle/input/datasets/cdeotte/s6e4-the-original-dataset/irrigation_prediction.csv")

In [35]:
train = train.drop("id", errors ="ignore", axis=1)

full_train = pd.concat([train, orig], ignore_index =True).drop_duplicates().reset_index(drop=True)
train = full_train.copy()

In [36]:
print(train.columns.tolist())

['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need']


In [37]:
train.dtypes
full_train.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [38]:
import optuna
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import minimize
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, TargetEncoder, OrdinalEncoder
from sklearn.metrics import balanced_accuracy_score

In [39]:
relevant_cols = ['Soil_Moisture', 'Temperature_C', 'Rainfall_mm', 'Wind_Speed_kmh', 'Crop_Growth_Stage', 'Mulching_Used', 'Irrigation_Need']
cols_to_drop = ['Irrigation_type', 'Water_Source']


train.drop(cols_to_drop, errors = 'ignore', inplace=True, axis = 1)
test.drop(cols_to_drop, errors = 'ignore', inplace=True, axis = 1)
x = train["Irrigation_Need"].value_counts()
for j,k in x.items():
    print(f"{j} is {k/train.shape[0]}")
print(x)

Low is 0.5871578125
Medium is 0.379490625
High is 0.0333515625
Irrigation_Need
Low       375781
Medium    242874
High       21345
Name: count, dtype: int64


In [40]:
cat = train.select_dtypes(include=['object','category'])
for col in cat:
    print(col,train[col].nunique())

Soil_Type 4
Crop_Type 6
Crop_Growth_Stage 4
Season 3
Irrigation_Type 4
Mulching_Used 2
Region 5
Irrigation_Need 3


# Feature Engineering

In [41]:
def engineer_features(df):
    df = df.copy()
    '''df['ET_proxy'] = df['Temperature_C'] * df['Sunlight_Hours'] / (df['Humidity'] + 1e-5)
    df['water_deficit'] = df['Temperature_C'] - df['Rainfall_mm']
    df['water_retention'] = df['Soil_Moisture'] * df['Organic_Carbon']
    df['heat_stress'] = df['Temperature_C'] / (df['Humidity'] + 1e-5)
    df['wind_evap'] = df['Wind_Speed_kmh'] * df['Temperature_C']
    df['salinity_stress'] = df['Electrical_Conductivity'] / (df['Soil_Moisture'] + 1e-5)
    df['prev_irrigation_per_area'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 1e-5) '''

    df["soil_lt_25"] = (df["Soil_Moisture"] < 25).astype(int)
    df["temp_gt_30"] = (df["Temperature_C"] > 30).astype(int)
    df["rain_lt_300"] = (df["Rainfall_mm"] < 300).astype(int)
    df["wind_gt_10"] = (df["Wind_Speed_kmh"] > 10).astype(int)
    return df
    
train = engineer_features(train)
test = engineer_features(test)

# Encoding

In [42]:
label_cols = [col for col in train.select_dtypes(include=['object','category']) if col != "Crop_Growth_Stage" and col != "Irrigation_Need"]
CGS_order = ['Sowing', 'Vegetative', 'Flowering', 'Harvest']

for cols in label_cols:
    le = LabelEncoder()
    train[cols] = le.fit_transform(train[cols])
    test[cols] = le.transform(test[cols])

le_target = LabelEncoder()
train['Irrigation_Need'] = le_target.fit_transform(train['Irrigation_Need'])

OE= OrdinalEncoder(categories = [CGS_order])
train["Crop_Growth_Stage"], test ["Crop_Growth_Stage"] = OE.fit_transform(train[["Crop_Growth_Stage"]]), OE.transform( test [["Crop_Growth_Stage"]])

In [43]:
X = train.drop(["id","Irrigation_Need"], errors = "ignore", axis = 1)
y = train["Irrigation_Need"]
X_train, X_test, y_train, y_test = train_test_split(X,y, stratify= y,random_state=42)

# Hyperparameter Tuning with Optuna

In [44]:
params =  {'objective':'multi:softmax',
            'eval_metric': 'mlogloss',
            'tree_method': 'hist',
            'device': 'cuda',
            'num_class': 3,
            'random_state': 42,
            'n_estimators': 3566, 'max_depth': 6, 'learning_rate': 0.010862098639117553, 'subsample': 0.7683625367134224, 'colsample_bytree': 0.6491368248904359, 'min_child_weight': 9, 'reg_alpha': 0.009208883541962158, 'reg_lambda': 0.021815878090553208, 'early_stopping_rounds':100}
prob_params =  {'objective':'multi:softprob',
            'eval_metric': 'mlogloss',
            'tree_method': 'hist',
            'device': 'cuda',
            'num_class': 3,
            'random_state': 42,
            'n_estimators': 3566, 'max_depth': 6, 'learning_rate': 0.010862098639117553, 'subsample': 0.7683625367134224, 'colsample_bytree': 0.6491368248904359, 'min_child_weight': 9, 'reg_alpha': 0.009208883541962158, 'reg_lambda': 0.021815878090553208, 'early_stopping_rounds':100}
model = XGBClassifier(**params)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model.fit(X_train, y_train, sample_weight= sample_weights,
         eval_set = [(X_test, y_test)],
         verbose = 100)

predictions = le_target.inverse_transform(model.predict(X_test))

[0]	validation_0-mlogloss:1.08483
[100]	validation_0-mlogloss:0.48106
[200]	validation_0-mlogloss:0.26590
[300]	validation_0-mlogloss:0.16859
[400]	validation_0-mlogloss:0.12366
[500]	validation_0-mlogloss:0.10210
[600]	validation_0-mlogloss:0.09095
[700]	validation_0-mlogloss:0.08452
[800]	validation_0-mlogloss:0.08031
[900]	validation_0-mlogloss:0.07704
[1000]	validation_0-mlogloss:0.07444
[1100]	validation_0-mlogloss:0.07264
[1200]	validation_0-mlogloss:0.07110
[1300]	validation_0-mlogloss:0.06982
[1400]	validation_0-mlogloss:0.06870
[1500]	validation_0-mlogloss:0.06778
[1600]	validation_0-mlogloss:0.06696
[1700]	validation_0-mlogloss:0.06622
[1800]	validation_0-mlogloss:0.06554
[1900]	validation_0-mlogloss:0.06491
[2000]	validation_0-mlogloss:0.06431
[2100]	validation_0-mlogloss:0.06377
[2200]	validation_0-mlogloss:0.06326
[2300]	validation_0-mlogloss:0.06282
[2400]	validation_0-mlogloss:0.06240
[2500]	validation_0-mlogloss:0.06199
[2600]	validation_0-mlogloss:0.06161
[2700]	valida

In [45]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probs = np.zeros((len(X_train), 3))
test_probs = np.zeros((len(X_test), 3))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\nFold {fold}/5")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    sample_weights = compute_sample_weight(class_weight='balanced', y=y_tr)
    model = XGBClassifier(**prob_params)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
        sample_weight=sample_weights
    )

    oof_probs[val_idx] = model.predict_proba(X_val)
    test_probs += model.predict_proba(X_test)

    fold_pred = np.argmax(oof_probs[val_idx], axis=1)
    fold_score = balanced_accuracy_score(y_val, fold_pred)
    fold_scores.append(fold_score)
    print(f"Fold {fold} Balanced Accuracy: {fold_score:.5f}")

test_probs = test_probs / 5

print(f"\nMean OOF Balanced Accuracy: {np.mean(fold_scores):.5f}")
print(f"Std: {np.std(fold_scores):.5f}")


def apply_thresholds(probs, t):
    adjusted = probs * np.array(t)
    return np.argmax(adjusted, axis=1)

def neg_balanced_acc(t):
    preds = apply_thresholds(oof_probs, t)
    return -balanced_accuracy_score(y_train, preds)

result = minimize(neg_balanced_acc, x0=[1, 1, 1], method='Nelder-Mead')
best_thresholds = result.x

print(f"\nBest thresholds: {best_thresholds}")
print(f"OOF Balanced Accuracy after tuning: {-result.fun:.5f}")

final_preds = apply_thresholds(test_probs, best_thresholds)


Fold 1/5
Fold 1 Balanced Accuracy: 0.97016

Fold 2/5
Fold 2 Balanced Accuracy: 0.96869

Fold 3/5
Fold 3 Balanced Accuracy: 0.96805

Fold 4/5
Fold 4 Balanced Accuracy: 0.96969

Fold 5/5
Fold 5 Balanced Accuracy: 0.97125

Mean OOF Balanced Accuracy: 0.96957
Std: 0.00112

Best thresholds: [1.40371792 1.14351387 0.38145424]
OOF Balanced Accuracy after tuning: 0.97311


In [46]:
'''def objective(trial):
    params = {
        'objective': 'multi:softmax',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'n_estimators': trial.suggest_int('n_estimators', 100, 3000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.9, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 5, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 5, log=True),
        'random_state': 42,
        'tree_method': 'hist',  # faster on large datasets
        'device': 'cuda'  # remove if no GPU
    }

    model = XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv,
        scoring='balanced_accuracy',
        params={'sample_weight': sample_weights}
    )

    return scores.mean()

study = optuna.create_study(direction='maximize',
    study_name="my_kaggle_study",
    storage="sqlite:///optuna_study.db",
    load_if_exists=True
)
study.optimize(objective, n_trials=75, show_progress_bar=True)

print(f"Best score: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")'''

'def objective(trial):\n    params = {\n        \'objective\': \'multi:softmax\',\n        \'num_class\': 3,\n        \'eval_metric\': \'mlogloss\',\n        \'n_estimators\': trial.suggest_int(\'n_estimators\', 100, 3000),\n        \'max_depth\': trial.suggest_int(\'max_depth\', 3, 10),\n        \'learning_rate\': trial.suggest_float(\'learning_rate\', 0.01, 0.9, log=True),\n        \'subsample\': trial.suggest_float(\'subsample\', 0.5, 1.0),\n        \'colsample_bytree\': trial.suggest_float(\'colsample_bytree\', 0.5, 1.0),\n        \'min_child_weight\': trial.suggest_int(\'min_child_weight\', 1, 10),\n        \'reg_alpha\': trial.suggest_float(\'reg_alpha\', 1e-8, 5, log=True),\n        \'reg_lambda\': trial.suggest_float(\'reg_lambda\', 1e-8, 5, log=True),\n        \'random_state\': 42,\n        \'tree_method\': \'hist\',  # faster on large datasets\n        \'device\': \'cuda\'  # remove if no GPU\n    }\n\n    model = XGBClassifier(**params)\n    cv = StratifiedKFold(n_splits=5, 

In [47]:
print(model.predict(X_test).shape)
print(X_train.shape)

(160000,)
(480000, 22)


In [51]:
y_pred = model.predict(X_test)
score = balanced_accuracy_score(y_test, y_pred)
print(f"Balanced Accuracy: {score:.4f}")

Balanced Accuracy: 0.9730


# Training with Full Dataset

In [49]:
best_n = model.best_iteration
best_n = int(best_n /0.75) #Scaling best_n by the percent of X_train samples to find optimal n estimators
params_full = {
    'objective': 'multi:softmax',
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda',
    'num_class': 3,
    'n_estimators': best_n,
    'max_depth': 3,
    'learning_rate': 0.1,
    'subsample': 0.81,
    'colsample_bytree': 0.61,
    'min_child_weight': 9,
    'reg_alpha': 1.4,
    'reg_lambda': 3.5
}

model_full = XGBClassifier(**params_full)
sample_weights = compute_sample_weight(class_weight='balanced', y=y)
model_full.fit(X, y, sample_weight=sample_weights)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.61, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=9, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=4753, n_jobs=None, num_class=3, ...)

# Submission

In [50]:
test_id = test['id']
test = test.copy().drop("id", axis=1)

preds = le_target.inverse_transform(model_full.predict(test))
submission = pd.DataFrame({'id': test_id,
                          'Irrigation_Need': preds})
submission.to_csv('submission.csv',index=False)